In [16]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load the Excel dataset
df = pd.read_excel("Lab 8.xlsx")
df = df.drop(columns=["No"])

print("Dataset preview:")
print(df.head().to_string(index=False))
print("\nDataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())

Dataset preview:
 Outlook Temperature Humidity   Wind Play Tennis
   Sunny         Hot     High   Weak          No
   Sunny         Hot     High Strong          No
Overcast         Hot     High   Weak         Yes
    Rain        Mild     High   Weak         Yes
    Rain        Cool   Normal   Weak         Yes

Dataset shape: (50, 5)

Columns: ['Outlook', 'Temperature', 'Humidity', 'Wind', 'Play Tennis']


In [17]:
# Data preprocessing for categorical learning
feature_columns = ["Outlook", "Temperature", "Humidity", "Wind"]
target_column = "Play Tennis"

feature_encoders = {}
X_encoded = pd.DataFrame(index=df.index)

for column in feature_columns:
    encoder = LabelEncoder()
    X_encoded[column] = encoder.fit_transform(df[column].astype(str))
    feature_encoders[column] = encoder

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df[target_column].astype(str))

print("Feature encodings:")
for column, encoder in feature_encoders.items():
    mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
    print(f"{column}: {mapping}")

print("\nTarget encoding:")
print(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

print("\nEncoded feature sample:")
print(X_encoded.head().to_string(index=False))
print("\nEncoded target sample:")
print(pd.Series(y_encoded).head().to_string(index=False))

Feature encodings:
Outlook: {'Overcast': np.int64(0), 'Rain': np.int64(1), 'Sunny': np.int64(2)}
Temperature: {'Cool': np.int64(0), 'Hot': np.int64(1), 'Mild': np.int64(2)}
Humidity: {'High': np.int64(0), 'Normal': np.int64(1)}
Wind: {'Strong': np.int64(0), 'Weak': np.int64(1)}

Target encoding:
{'No': np.int64(0), 'Yes': np.int64(1)}

Encoded feature sample:
 Outlook  Temperature  Humidity  Wind
       2            1         0     1
       2            1         0     0
       0            1         0     1
       1            2         0     1
       1            0         1     1

Encoded target sample:
0
0
1
1
1


In [18]:
# Dataset partitioning
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y_encoded,
    test_size=0.30,
    random_state=42,
    stratify=y_encoded,
)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

# Naive Bayes model training and evaluation
nb_model = CategoricalNB(alpha=1.0)
nb_model.fit(X_train, y_train)

nb_predictions = nb_model.predict(X_test)
nb_probabilities = nb_model.predict_proba(X_test)

print("Naive Bayes Evaluation")
print("Accuracy:", round(accuracy_score(y_test, nb_predictions), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, nb_predictions))
print("\nClassification Report:")
print(classification_report(y_test, nb_predictions, target_names=label_encoder.classes_, zero_division=0))

Training set size: (35, 4)
Testing set size: (15, 4)
Naive Bayes Evaluation
Accuracy: 0.8667

Confusion Matrix:
[[ 3  2]
 [ 0 10]]

Classification Report:
              precision    recall  f1-score   support

          No       1.00      0.60      0.75         5
         Yes       0.83      1.00      0.91        10

    accuracy                           0.87        15
   macro avg       0.92      0.80      0.83        15
weighted avg       0.89      0.87      0.86        15



In [19]:
# Single-sample inference
custom_sample = pd.DataFrame(
    [
        {
            "Outlook": "Sunny",
            "Temperature": "Cool",
            "Humidity": "High",
            "Wind": "Strong",
        }
    ]
)

custom_sample_encoded = custom_sample.copy()
for column in feature_columns:
    custom_sample_encoded[column] = feature_encoders[column].transform(custom_sample[column].astype(str))

predicted_class_encoded = nb_model.predict(custom_sample_encoded)[0]
predicted_class = label_encoder.inverse_transform([predicted_class_encoded])[0]
predicted_probabilities = nb_model.predict_proba(custom_sample_encoded)[0]
probability_table = pd.DataFrame(
    {
        "Class": label_encoder.classes_,
        "Probability": predicted_probabilities,
    }
)

print("Custom query:")
print(custom_sample.to_string(index=False))
print("\nPredicted class:", predicted_class)
print("\nPrediction probabilities:")
print(probability_table.to_string(index=False))

Custom query:
Outlook Temperature Humidity   Wind
  Sunny        Cool     High Strong

Predicted class: No

Prediction probabilities:
Class  Probability
   No     0.746606
  Yes     0.253394


In [20]:
# Model comparison on the same encoded train/test split
models = {
    "Categorical Naive Bayes": CategoricalNB(alpha=1.0),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM": SVC(kernel="rbf", random_state=42),
}

comparison_rows = []
for model_name, model in models.items():
    model.fit(X_train, y_train)
    model_predictions = model.predict(X_test)
    comparison_rows.append(
        {
            "Model": model_name,
            "Accuracy": round(accuracy_score(y_test, model_predictions), 4),
            "Confusion Matrix": confusion_matrix(y_test, model_predictions).tolist(),
            "Predicted Classes": label_encoder.inverse_transform(model_predictions).tolist(),
        }
    )

comparison_df = pd.DataFrame(comparison_rows).sort_values(by="Accuracy", ascending=False).reset_index(drop=True)
print("Model comparison table:")
print(comparison_df.to_string(index=False))

Model comparison table:
                  Model  Accuracy  Confusion Matrix                                                        Predicted Classes
          Decision Tree    1.0000 [[5, 0], [0, 10]]   [Yes, No, Yes, No, Yes, Yes, Yes, No, Yes, Yes, Yes, Yes, No, Yes, No]
    Logistic Regression    0.9333 [[4, 1], [0, 10]]  [Yes, No, Yes, No, Yes, Yes, Yes, Yes, Yes, Yes, Yes, Yes, No, Yes, No]
                    SVM    0.9333 [[4, 1], [0, 10]]  [Yes, No, Yes, No, Yes, Yes, Yes, Yes, Yes, Yes, Yes, Yes, No, Yes, No]
Categorical Naive Bayes    0.8667 [[3, 2], [0, 10]] [Yes, No, Yes, No, Yes, Yes, Yes, Yes, Yes, Yes, Yes, Yes, Yes, Yes, No]


## Analysis Report

The models produce different predictions because they learn from the same categorical data in different ways. `CategoricalNB` estimates class probabilities directly from category counts and assumes the predictors are conditionally independent, so its scores reflect the observed frequency patterns in the training set. `DecisionTreeClassifier` can capture interaction rules between features, while `LogisticRegression` and `SVC` treat the label-encoded categories as numeric inputs, which introduces an artificial ordering that changes their decision boundaries and probability estimates. Because the dataset is very small, those modeling assumptions have a visible effect on both the predicted class and the probability scores for the same test instance.

In [21]:
# Data preprocessing for categorical learning
feature_columns = ["Outlook", "Temperature", "Humidity", "Wind"]
target_column = "Play Tennis"

feature_encoders = {}
X_encoded = pd.DataFrame(index=df.index)

for column in feature_columns:
    encoder = LabelEncoder()
    X_encoded[column] = encoder.fit_transform(df[column].astype(str))
    feature_encoders[column] = encoder

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(df[target_column].astype(str))

print("Feature encodings:")

Feature encodings:
